# Rally financiero 1 — análisis de series de tiempo

**Objetivo.** Analizar el comportamiento del USD/MXN y medir su sensibilidad ante movimientos del mercado estadounidense. Se utilizan el S&P 500, el VIX y el rendimiento del Treasury a 10 años como variables financieras de referencia.

El ejercicio combina análisis descriptivo, estacionariedad, modelos ARIMA/SARIMAX y una regresión con errores HAC. La validación se realiza fuera de muestra para evitar evaluar el modelo con los mismos datos usados en la estimación.

In [ ]:
!pip -q install yfinance

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
import statsmodels.api as sm

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 1. Datos de mercado
Se trabaja con precios de cierre diarios. El periodo termina el 24 de septiembre de 2026 para conservar el mismo corte utilizado en el ejercicio financiero.

In [ ]:
START = '2020-01-01'
END = '2026-09-25'  # yfinance usa el límite superior como fecha exclusiva

tickers = ['MXN=X', '^GSPC', '^VIX', '^TNX']
names = {
    'MXN=X': 'USDMXN',
    '^GSPC': 'SP500',
    '^VIX': 'VIX',
    '^TNX': 'TNX10Y'
}

raw = yf.download(
    tickers,
    start=START,
    end=END,
    auto_adjust=False,
    progress=False
)

if isinstance(raw.columns, pd.MultiIndex):
    field = 'Adj Close' if 'Adj Close' in raw.columns.get_level_values(0) else 'Close'
    prices = raw[field].copy()
else:
    prices = raw[['Adj Close']].copy()

prices = prices.rename(columns=names).sort_index()
prices = prices[['USDMXN', 'SP500', 'VIX', 'TNX10Y']].ffill().dropna()
prices.tail()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col in zip(axes.ravel(), prices.columns):
    ax.plot(prices.index, prices[col], linewidth=1)
    ax.set_title(col)
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 2. Transformación de las series
Los precios y niveles suelen presentar tendencia. Para el análisis econométrico se transforman a cambios diarios: rendimientos logarítmicos para USD/MXN y S&P 500, cambio absoluto del VIX y cambio en puntos porcentuales del Treasury a 10 años.

In [ ]:
data = pd.DataFrame(index=prices.index)
data['usdmxn_ret'] = 100 * np.log(prices['USDMXN']).diff()
data['sp500_ret'] = 100 * np.log(prices['SP500']).diff()
data['vix_change'] = prices['VIX'].diff()
data['tnx_change'] = prices['TNX10Y'].diff()
data = data.dropna()

data.describe().T

In [ ]:
corr = data.corr()
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr.values, aspect='auto')
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr.index)), corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center')
ax.set_title('Correlaciones entre cambios diarios')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 3. Prueba de estacionariedad
La prueba Dickey-Fuller aumentada contrasta la hipótesis nula de raíz unitaria. Un valor p menor a 0.05 permite rechazar esa hipótesis al 5%.

In [ ]:
def adf_table(series, name):
    result = adfuller(series.dropna(), autolag='AIC')
    return {
        'serie': name,
        'ADF': result[0],
        'p_value': result[1],
        'rezagos': result[2],
        'n_obs': result[3]
    }

tests = [
    adf_table(prices['USDMXN'], 'USD/MXN nivel'),
    adf_table(data['usdmxn_ret'], 'USD/MXN rendimiento'),
    adf_table(prices['SP500'], 'S&P 500 nivel'),
    adf_table(data['sp500_ret'], 'S&P 500 rendimiento')
]
pd.DataFrame(tests).set_index('serie')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(data['usdmxn_ret'], lags=30, zero=False, ax=axes[0])
plot_pacf(data['usdmxn_ret'], lags=30, zero=False, method='ywm', ax=axes[1])
axes[0].set_title('ACF — rendimiento USD/MXN')
axes[1].set_title('PACF — rendimiento USD/MXN')
plt.tight_layout()
plt.show()

## 4. Separación temporal y benchmark
El último 20% de las observaciones se reserva para evaluación fuera de muestra. Como referencia se utiliza un pronóstico ingenuo igual a cero para el rendimiento diario.

In [ ]:
split = int(len(data) * 0.80)
train = data.iloc[:split].copy()
test = data.iloc[split:].copy()

y_train = train['usdmxn_ret']
y_test = test['usdmxn_ret']

def metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5
    }

benchmark = pd.Series(0.0, index=y_test.index)
benchmark_metrics = metrics(y_test, benchmark)
benchmark_metrics

## 5. ARIMA univariado
Se seleccionan los órdenes AR y MA mediante AIC dentro de una cuadrícula pequeña. Como la variable dependiente ya es un rendimiento estacionario, se fija d = 0.

In [ ]:
candidates = []
for p in range(4):
    for q in range(4):
        if p == 0 and q == 0:
            continue
        try:
            fit = ARIMA(y_train, order=(p, 0, q), trend='c').fit()
            candidates.append((p, q, fit.aic))
        except Exception:
            pass

aic_table = pd.DataFrame(candidates, columns=['p', 'q', 'AIC']).sort_values('AIC')
aic_table.head(10)

In [ ]:
best_p = int(aic_table.iloc[0]['p'])
best_q = int(aic_table.iloc[0]['q'])

arima = ARIMA(y_train, order=(best_p, 0, best_q), trend='c').fit()
arima_pred = pd.Series(arima.forecast(steps=len(y_test)).values, index=y_test.index)
arima_metrics = metrics(y_test, arima_pred)

print(f'ARIMA seleccionado: ({best_p}, 0, {best_q})')
print(arima_metrics)

## 6. SARIMAX con factores financieros externos
El modelo incorpora el rendimiento del S&P 500, el cambio del VIX y el cambio del Treasury a 10 años. Esto permite evaluar si la información internacional mejora el pronóstico del USD/MXN respecto al modelo puramente autorregresivo.

In [ ]:
xcols = ['sp500_ret', 'vix_change', 'tnx_change']
X_train = train[xcols]
X_test = test[xcols]

sarimax = SARIMAX(
    y_train,
    exog=X_train,
    order=(best_p, 0, best_q),
    trend='c',
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

sarimax_pred = sarimax.get_forecast(steps=len(y_test), exog=X_test).predicted_mean
sarimax_pred.index = y_test.index
sarimax_metrics = metrics(y_test, sarimax_pred)
sarimax_metrics

In [ ]:
comparison = pd.DataFrame([
    {'modelo': 'Benchmark cero', **benchmark_metrics},
    {'modelo': f'ARIMA({best_p},0,{best_q})', **arima_metrics},
    {'modelo': 'SARIMAX + factores externos', **sarimax_metrics}
]).set_index('modelo')
comparison

In [ ]:
window = min(160, len(y_test))
plt.figure(figsize=(13, 5))
plt.plot(y_test.index[-window:], y_test.iloc[-window:], label='Observado', linewidth=1.2)
plt.plot(arima_pred.index[-window:], arima_pred.iloc[-window:], label='ARIMA', linewidth=1)
plt.plot(sarimax_pred.index[-window:], sarimax_pred.iloc[-window:], label='SARIMAX', linewidth=1)
plt.axhline(0, linewidth=0.8)
plt.title('Validación fuera de muestra — rendimiento diario USD/MXN')
plt.ylabel('% diario')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 7. Sensibilidad contemporánea del USD/MXN
Además del pronóstico se estima una regresión lineal con errores estándar HAC (Newey-West). El coeficiente del S&P 500 puede interpretarse como la sensibilidad promedio del rendimiento diario del USD/MXN ante un movimiento de 1% del índice, manteniendo constantes los cambios del VIX y del Treasury.

In [ ]:
reg = data.copy()
reg['usdmxn_lag1'] = reg['usdmxn_ret'].shift(1)
reg = reg.dropna()

X = sm.add_constant(reg[['sp500_ret', 'vix_change', 'tnx_change', 'usdmxn_lag1']])
y = reg['usdmxn_ret']

ols_hac = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
print(ols_hac.summary())

In [ ]:
beta_sp = ols_hac.params['sp500_ret']
p_sp = ols_hac.pvalues['sp500_ret']
beta_vix = ols_hac.params['vix_change']
beta_tnx = ols_hac.params['tnx_change']

print(f'Sensibilidad al S&P 500: {beta_sp:.4f}')
print(f'p-value S&P 500: {p_sp:.4g}')
print(f'Sensibilidad al cambio del VIX: {beta_vix:.4f}')
print(f'Sensibilidad al Treasury 10Y: {beta_tnx:.4f}')

if beta_sp < 0:
    print('Lectura: en promedio, una caída del S&P 500 se asocia con presión de depreciación del peso frente al dólar.')
else:
    print('Lectura: en esta muestra, el coeficiente estimado del S&P 500 es positivo.')

## 8. Beta móvil de mercado
Para revisar si la relación cambia con el tiempo se calcula una beta móvil de 60 sesiones entre el rendimiento del USD/MXN y el S&P 500. No es una beta CAPM de una acción: aquí funciona como una medida descriptiva de sensibilidad cambiaria frente al mercado.

In [ ]:
rolling_cov = data['usdmxn_ret'].rolling(60).cov(data['sp500_ret'])
rolling_var = data['sp500_ret'].rolling(60).var()
rolling_beta = rolling_cov / rolling_var

plt.figure(figsize=(13, 4))
plt.plot(rolling_beta.index, rolling_beta, linewidth=1)
plt.axhline(0, linewidth=0.8)
plt.title('Sensibilidad móvil USD/MXN vs. S&P 500 — 60 sesiones')
plt.ylabel('Beta móvil')
plt.grid(alpha=0.25)
plt.show()

rolling_beta.dropna().describe()

## 9. Pronóstico univariado de corto plazo
Para un pronóstico estrictamente ex ante no se conocen todavía los valores futuros del S&P 500, VIX y Treasury. Por eso el cierre utiliza el ARIMA univariado para proyectar cinco sesiones.

In [ ]:
final_arima = ARIMA(data['usdmxn_ret'], order=(best_p, 0, best_q), trend='c').fit()
forecast = final_arima.get_forecast(steps=5)
forecast_table = forecast.summary_frame(alpha=0.05)[['mean', 'mean_ci_lower', 'mean_ci_upper']]
forecast_table.columns = ['pronostico', 'limite_inferior_95', 'limite_superior_95']
forecast_table

## Conclusión del ejercicio
La comparación principal no debe basarse únicamente en ajuste dentro de muestra. El criterio central es el error fuera de muestra. Si el SARIMAX reduce MAE y RMSE respecto al ARIMA y al benchmark, existe evidencia de que los factores internacionales aportan información útil para explicar la dinámica diaria del USD/MXN.

La regresión HAC complementa esa comparación al cuantificar la dirección y magnitud de la relación contemporánea. Un coeficiente negativo y estadísticamente significativo para el S&P 500 implica que jornadas negativas del mercado estadounidense tienden a coincidir con aumentos del USD/MXN, es decir, depreciación del peso. Esto conecta el riesgo bursátil internacional con el riesgo cambiario de una empresa mexicana que tiene obligaciones denominadas en dólares.